In [1]:
# 약물리스트 추출

import xml.etree.ElementTree as ET
import pandas as pd

# 1. 파일 설정
xml_file = 'full database.xml'
output_csv = 'drug_information/drugbank_all_drugs_list.csv'

# 2. 네임스페이스 (XSD 기준)
ns = {'db': 'http://www.drugbank.ca'}

def extract_text(element, path):
    """태그 텍스트 안전하게 추출"""
    found = element.find(path, ns)
    return found.text if found is not None else None

print(f"전체 약물 리스트 추출 시작... ({xml_file})")

rows = []
context = ET.iterparse(xml_file, events=('end',))

for event, elem in context:
    tag = elem.tag.replace(f"{{{ns['db']}}}", "")
    
    # <drug> 태그가 끝날 때마다 정보 수집
    if event == 'end' and tag == 'drug':
        # type 속성이 없으면 중첩된 태그이므로 건너뛰기 (루트 레벨 drug만 처리)
        if 'type' not in elem.attrib:
            elem.clear()
            continue
        
        try:
            # --- 기본 식별 정보 ---
            # 1. DrugBank ID (Primary)
            drug_id_elem = elem.find("db:drugbank-id[@primary='true']", ns)
            drug_id = drug_id_elem.text if drug_id_elem is not None else None
            
            # 2. 약물 이름
            name = extract_text(elem, "db:name")
            
            # 3. 약물 타입 (Small Molecule / Biotech)
            drug_type = elem.attrib.get('type')
            
            # 4. CAS Number (화학 식별 번호)
            cas_number = extract_text(elem, "db:cas-number")
            
            # --- 상태 정보 ---
            # 5. 그룹 (Approved, Investigational 등)
            groups = [g.text for g in elem.findall("db:groups/db:group", ns)]
            groups_str = "; ".join(groups) if groups else ""
            
            # --- 구조 정보 ---
            # 6. SMILES (구조식)
            smiles = None
            if drug_type == 'small molecule':
                for prop in elem.findall("db:calculated-properties/db:property", ns):
                    kind = prop.find("db:kind", ns)
                    if kind is not None and kind.text == "SMILES":
                        smiles = prop.find("db:value", ns).text
                        break
            
            # --- 설명 정보 ---
            # 7. Description (한 줄 요약)
            description = extract_text(elem, "db:description")
            if description:
                description = description[:200].replace('\n', ' ') + "..." # 너무 기니까 앞부분만

            # 리스트에 추가 (snake_case)
            rows.append({
                'drugbank_id': drug_id,
                'name': name,
                'type': drug_type,
                'cas_number': cas_number,
                'groups': groups_str,
                'smiles': smiles,
                'description': description
            })

        except Exception as e:
            print(f"Error: {e}")
        
        # 메모리 청소 (필수)
        elem.clear()

# 3. CSV 저장
df_all_drugs = pd.DataFrame(rows)
print(f"총 추출된 약물 개수: {len(df_all_drugs)}")

df_all_drugs.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"저장 완료: {output_csv}")
print("-" * 50)
print(df_all_drugs.head())

전체 약물 리스트 추출 시작... (full database.xml)
총 추출된 약물 개수: 17430
저장 완료: drugbank_all_drugs_list.csv
--------------------------------------------------
  drugbank_id                 name     type   cas_number  \
0     DB00001            Lepirudin  biotech  138068-37-8   
1     DB00002            Cetuximab  biotech  205923-56-4   
2     DB00003         Dornase alfa  biotech  143831-71-4   
3     DB00004  Denileukin diftitox  biotech  173146-27-5   
4     DB00005           Etanercept  biotech  185243-69-0   

                      groups smiles  \
0        approved; withdrawn   None   
1                   approved   None   
2                   approved   None   
3  approved; investigational   None   
4  approved; investigational   None   

                                         description  
0  Lepirudin is a recombinant hirudin formed by 6...  
1  Cetuximab is a recombinant chimeric human/mous...  
2  Dornase alfa is a biosynthetic form of human d...  
3  Denileukin diftitox is an IL2-recepto

In [2]:
df_all_drugs

,drugbank_id,name,type,cas_number,groups,smiles,description
0,DB00001,Lepirudin,biotech,138068-37-8,approved; withdrawn,None,Lepirudin is a recombinant hirudin formed by 6...
1,DB00002,Cetuximab,biotech,205923-56-4,approved,None,Cetuximab is a recombinant chimeric human/mous...
2,DB00003,Dornase alfa,biotech,143831-71-4,approved,None,Dornase alfa is a biosynthetic form of human d...
3,DB00004,Denileukin diftitox,biotech,173146-27-5,approved; investigational,None,Denileukin diftitox is an IL2-receptor-directe...
4,DB00005,Etanercept,biotech,185243-69-0,approved; investigational,None,Dimeric fusion protein consisting of the extra...
...,...,...,...,...,...,...,...
17425,DB19452,Exidavnemab,biotech,2438229-02-6,investigational,None,Exidavnemab is under investigation in clinical...
17426,DB19453,Imciromab pentetate,biotech,138660-99-8,investigational,None,None
17427,DB19454,Cetyl oleate,small molecule,22393-86-8,investigational,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC\C=C/CCCCCCCC,None
17428,DB19455,Cetyl myristoleate,small molecule,64660-84-0,investigational,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC\C=C/CCCC,None


In [3]:
df_all_drugs["type"].value_counts()

type
small molecule    13166
biotech            4264
Name: count, dtype: int64

In [4]:
import pandas as pd

# (만약 커널을 재시작했다면) 저장한 CSV 다시 불러오기
# df_all_drugs = pd.read_csv('drugbank_all_drugs_list.csv')

print(f"전체 데이터 개수: {len(df_all_drugs)} 개\n")

print("--- [컬럼별 결측치(NaN) 개수] ---")
print(df_all_drugs.isnull().sum())

print("\n--- [컬럼별 결측치 비율 (%)] ---")
missing_ratio = (df_all_drugs.isnull().sum() / len(df_all_drugs)) * 100
print(missing_ratio.round(2))

전체 데이터 개수: 17430 개

--- [컬럼별 결측치(NaN) 개수] ---
drugbank_id       0
name              0
type              0
cas_number     6663
groups            0
smiles         5117
description    7405
dtype: int64

--- [컬럼별 결측치 비율 (%)] ---
drugbank_id     0.00
name            0.00
type            0.00
cas_number     38.23
groups          0.00
smiles         29.36
description    42.48
dtype: float64


In [8]:
df_all_drugs.to_csv("drug_information/drug_information.csv", index=False)

In [5]:
# 단백질 의약품은 smiles 정보가 없음
bio = df_all_drugs[df_all_drugs["type"] == "biotech"] 
bio
bio["smiles"].isnull().sum()

np.int64(4264)

In [6]:
in_smiles = df_all_drugs[df_all_drugs["smiles"].notnull()]
in_smiles

,drugbank_id,name,type,cas_number,groups,smiles,description
5,DB00006,Bivalirudin,small molecule,128270-60-0,approved; investigational,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,Bivalirudin is a synthetic 20 residue peptide ...
13,DB00014,Goserelin,small molecule,65807-02-5,approved,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,"Goserelin is a synthetic hormone. In men, it s..."
25,DB00027,Gramicidin D,small molecule,1405-97-6,approved,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,Gramcidin D is a heterogeneous mixture of thre...
33,DB00035,Desmopressin,small molecule,16679-58-6,approved,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,"Desmopressin (dDAVP), a synthetic analogue of ..."
47,DB00050,Cetrorelix,small molecule,120287-85-6,approved; investigational,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...,Cetrorelix is a man-made hormone that blocks t...
...,...,...,...,...,...,...,...
17421,DB19446,Tulmimetostat,small molecule,2567686-02-4,investigational,[H][C@@]1(CC[C@@H](CC1)N1CC(C1)OC)[C@@]1(C)OC2...,Tulmimetostat is under investigation in clinic...
17424,DB19451,Ibuzatrelvir,small molecule,2755812-39-4,investigational,COC(=O)N[C@H](C(=O)N1C[C@@H](C[C@H]1C(=O)N[C@@...,Ibuzatrelvir is under investigation in clinica...
17427,DB19454,Cetyl oleate,small molecule,22393-86-8,investigational,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC\C=C/CCCCCCCC,None
17428,DB19455,Cetyl myristoleate,small molecule,64660-84-0,investigational,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC\C=C/CCCC,None


In [7]:
in_smiles["groups"].value_counts()

groups
experimental                                              5826
investigational                                           3321
approved                                                  1029
approved; investigational                                  951
approved; experimental                                     134
approved; vet_approved                                     127
experimental; investigational                               99
experimental; illicit                                       95
approved; withdrawn                                         93
vet_approved                                                91
approved; investigational; vet_approved                     75
withdrawn                                                   61
approved; investigational; withdrawn                        55
experimental; vet_approved                                  47
approved; illicit                                           38
approved; nutraceutical                         

In [ ]:
in_smiles = in_smiles.drop(columns=["description", "cas_number"])

In [ ]:
in_smiles["type"].value_counts()

In [24]:
in_smiles

,DrugBank_ID,Name,Type,Groups,SMILES
12,DB00006,Bivalirudin,small molecule,approved; investigational,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...
26,DB00014,Goserelin,small molecule,approved,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
41,DB00027,Gramicidin D,small molecule,approved,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...
55,DB00035,Desmopressin,small molecule,approved,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...
69,DB00050,Cetrorelix,small molecule,approved; investigational,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...
...,...,...,...,...,...
73678,DB19446,Tulmimetostat,small molecule,investigational,[H][C@@]1(CC[C@@H](CC1)N1CC(C1)OC)[C@@]1(C)OC2...
73681,DB19451,Ibuzatrelvir,small molecule,investigational,COC(=O)N[C@H](C(=O)N1C[C@@H](C[C@H]1C(=O)N[C@@...
73684,DB19454,Cetyl oleate,small molecule,investigational,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC\C=C/CCCCCCCC
73685,DB19455,Cetyl myristoleate,small molecule,investigational,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC\C=C/CCCC
